# 02 · Agregações de Negócio (Caso A)

**Teoria**: docs/04-dataframes-catalyst-tungsten.md

Ainda `local[*]`. Este notebook fica só em `vendas` — sem join ainda —
para consolidar `groupBy`/`agg`/`orderBy` antes de complicar com
múltiplas tabelas (isso vem no notebook 03).

In [ ]:
import sys

sys.path.insert(0, "../scripts")
from lab_utils import get_local_session, layer_path

spark = get_local_session("02-agregacoes-negocio")
vendas = spark.read.parquet(layer_path("local", "bronze", "vendas"))
vendas.show(5)

## Total de vendas por ano/mês

Como a receita evoluiu mês a mês?

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.functions import sum as spark_sum

vendas_por_periodo = (
    vendas.groupBy("ano", "mes")
    .agg(spark_sum("valor").alias("total_vendas"))
    .orderBy("ano", "mes")
)
vendas_por_periodo.show(24)

## Ticket médio e volume de transações

In [ ]:
from pyspark.sql.functions import avg, count

vendas.agg(
    avg("valor").alias("ticket_medio"),
    count("*").alias("total_transacoes"),
).show()

## As 10 maiores vendas individuais

In [ ]:
maiores_vendas = (
    vendas.select("id_venda", "id_funcionario", "valor", "ano", "mes", "dia")
    .orderBy(col("valor").desc())
)
maiores_vendas.show(10)

## Vendas por ano: total e contagem juntos

`agg` aceita várias agregações de uma vez — não precisa de uma chamada por
métrica.

In [ ]:
resumo_anual = (
    vendas.groupBy("ano")
    .agg(
        spark_sum("valor").alias("total_vendas"),
        count("*").alias("numero_vendas"),
        avg("valor").alias("ticket_medio"),
    )
    .orderBy("ano")
)
resumo_anual.show()

In [ ]:
spark.stop()